# Credit Default Model (ฉบับจัดระเบียบ)

**วิธีใช้:** แก้ค่าใน **Settings** (cell เดียว) → กด Run All → ส่งไฟล์ตามที่สรุปไว้ท้าย notebook

| ขั้นตอน | ทำอะไร |
|---|---|
| 0. Settings | ตั้ง path, ชื่อรอบ, สวิตช์ต่างๆ |
| 1. ข้อมูล | โหลด + ตรวจความถูกต้อง |
| 2. Features | สร้าง ratio / frequency features (ไม่มี leakage) |
| 3. โมเดล + CV | Stratified 5-fold, fold เดียวกันทุกการทดลอง |
| 4. ทดลอง | เทียบ raw vs +engineered (+ablation, +CatBoost ถ้าเปิด) |
| 5. Threshold | เลือกจุดตัด 0/1 จาก OOF (Youden's J) |
| 6. Submission | สร้างไฟล์ probability และไฟล์ 0/1 |

## 0. Settings
ทุกอย่างที่ต้องแก้อยู่ใน cell นี้ cell เดียว

In [42]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve
import lightgbm as lgb

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:          # รันนอก Jupyter
    display = print

# ---------- ไฟล์และชื่อรอบ ----------
DATA_DIR = Path(r"C:\Users\ice55\Downloads\Aihack Thailand 2026\data")   # <-- แก้ path ตรงนี้
RUN_NAME = "v7"              # ชื่อรอบ -> ไฟล์ผลลัพธ์จะเป็น submission_proba_v4.csv ฯลฯ (เปลี่ยนทุกครั้งกันเขียนทับ)
TARGET   = "12-Month Default Flag"

# ---------- สวิตช์ ----------
USE_CATBOOST   = False       # True = รัน CatBoost แล้วเฉลี่ยกับ LGBM (ต้อง pip install catboost)
USE_GENDER     = False       # ปิดไว้: credit scoring ปกติเลี่ยงเพศ
USE_ID_FEATURE = False       # เปิดได้เมื่อกติกาอนุญาต (ดู cell ตรวจ ID ในหัวข้อ 1)
RUN_ABLATION   = False       # True = ทดสอบ feature ทีละกลุ่ม (ช้ากว่า ~5 เท่า)

# ---------- Cross-validation ----------
SEED    = 42
N_FOLDS = 10

if USE_CATBOOST:
    from catboost import CatBoostClassifier

## 1. ข้อมูล
โหลดไฟล์ + ตรวจว่า ID ใน template ตรงกับ test

In [43]:
train    = pd.read_csv(DATA_DIR / "train_dataset.csv")
test     = pd.read_csv(DATA_DIR / "test_without_gt.csv")
template = pd.read_csv(DATA_DIR / "submission_template_for_test.csv")

assert train["ID"].is_unique and test["ID"].is_unique
assert set(template["ID"]) == set(test["ID"]), "ID ใน template ไม่ตรงกับ test"

print(f"train {train.shape} | test {test.shape} | template {template.shape}")
print(f"default rate (train) = {train[TARGET].mean():.4f}")

missing = train.isna().sum()
print("\ncolumns ที่มี missing:", "ไม่มี" if missing.sum() == 0 else "")
if missing.sum() > 0:
    print(missing[missing > 0].to_string())

train (68546, 27) | test (137091, 26) | template (137091, 2)
default rate (train) = 0.0968

columns ที่มี missing: ไม่มี


**ตรวจหน่วยของตัวเลขเงิน** (data dict บอก ¥10,000 / ¥1,000 แต่ค่าที่เห็นดูเป็นเยนตรงๆ)
ไม่กระทบ tree เพราะ ratio ต่างกันแค่ค่าคงที่ แต่ถ้าจะตีความตัวเลขเอง ต้องใช้หน่วยที่ตรวจได้จากตารางนี้

In [44]:
money_cols = ["Declared Annual Income", "Declared Borrowing Amount", "Desired Credit Limit", "Rent Burden"]
summary = train[money_cols].describe().T[["min", "50%", "max"]]
summary["zero_share"] = (train[money_cols] == 0).mean()
display(summary.round(3))

,min,50%,max,zero_share
Declared Annual Income,0.0,3000000.0,9.480000e+10,0.000
Declared Borrowing Amount,0.0,300000.0,1.236661e+10,0.370
Desired Credit Limit,10000.0,200000.0,8.000000e+06,0.000
Rent Burden,0.0,10000.0,7.000000e+08,0.433


**(ทางเลือก) ตรวจ ID:** แบ่ง train เป็น 10 กลุ่มตามลำดับ ID ถ้า default rate ต่างกันชัด ID อาจเป็น proxy ของเวลา
ต้องเช็กกติกาก่อนจะเปิด `USE_ID_FEATURE`

In [45]:
id_decile = pd.qcut(train["ID"].rank(method="first"), 10, labels=False)
display(train.groupby(id_decile)[TARGET].agg(default_rate="mean", n="size").round(4))

,default_rate,n
ID,,
0,0.0977,6855
1,0.0961,6855
2,0.0970,6854
3,0.0910,6855
4,0.0919,6854
5,0.0944,6855
6,0.1037,6854
7,0.0954,6855
8,0.0986,6854


## 2. Features
ทุก feature คำนวณจากแถวของตัวเองหรือสถิติไม่ใช้ label -> ไม่มี leakage

หมายเหตุ: tree ไม่สนใจ log / Box-Cox / เปลี่ยนหน่วยของตัวแปรเดี่ยว จึงไม่ใส่ ที่ช่วยได้คือ **ratio, interaction, frequency**
ตัวหารที่เป็น 0 จะได้ `NaN` (LightGBM จัดการเอง) ไม่ใช่ `inf`

In [46]:
# คอลัมน์ที่เป็น "รหัสหมวดหมู่" (ไม่มีลำดับ) -> ให้โมเดลเห็นเป็น category
CAT_COLS = [
    "Application Channel", "Marital Status", "Occupation Category", "Industry Category",
    "Employment Type Category", "Insurance Type", "Business Entity Type",
    "Residence Type", "Residence Ownership Status",
]
if USE_GENDER:
    CAT_COLS.append("Gender")

PHONE_COLS = ["Home Phone Number Available", "Mobile Phone Number Available",
              "Employer Phone Number Available", "Driver Licence Number Available"]


def safe_div(a, b):
    b = b.astype(float)
    return a / b.where(b > 0)            # ตัวหาร <= 0 -> NaN


def make_features(df):
    d = df.copy()
    income = d["Declared Annual Income"]
    debt   = d["Declared Borrowing Amount"]
    limit  = d["Desired Credit Limit"]
    n_dep  = d["Number of Dependents"]

    # หนี้เทียบรายได้
    d["borrow_to_income"]   = safe_div(debt, income)
    d["limit_to_income"]    = safe_div(limit, income)
    d["exposure_to_income"] = safe_div(debt + limit, income)      # (หนี้เดิม + วงเงินที่ขอ) / รายได้
    d["limit_to_borrow"]    = safe_div(limit, debt)
    d["avg_loan"]           = safe_div(debt, d["Declared Number of Borrowings"])

    # ภาระที่อยู่อาศัยและครอบครัว
    d["rent_to_income"]       = safe_div(d["Rent Burden"], income)
    d["income_per_dependent"] = income / (n_dep + 1)
    d["children_share"]       = safe_div(d["Number of Dependent Children"], n_dep)

    # ความมั่นคงและข้อมูลที่ให้
    d["residence_ratio"] = safe_div(d["Residence Period (in months)"], d["Age (in months)"])
    d["phone_count"]     = d[PHONE_COLS].sum(axis=1)
    return d


def add_freq_features(d):
    # ความถี่ของคู่หมวดหมู่ (นับจาก train+test รวมกัน ไม่ใช้ label)
    pairs = {
        "occ_x_ind":     ("Occupation Category", "Industry Category"),
        "size_x_entity": ("Company Size Category", "Business Entity Type"),
        "occ_x_emp":     ("Occupation Category", "Employment Type Category"),
    }
    for name, (a, b) in pairs.items():
        key = d[a].astype(str) + "_" + d[b].astype(str)
        d[name + "_freq"] = key.map(key.value_counts(normalize=True))
    return d

In [47]:
# รวม train+test เพื่อให้ categories และ frequency ตรงกัน
full = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
full = add_freq_features(make_features(full))
if USE_ID_FEATURE:
    full["id_prefix"] = full["ID"] // 10000
for c in CAT_COLS:
    full[c] = full[c].astype("category")

# รายชื่อ feature (ENG_FEATURES สร้างอัตโนมัติจากคอลัมน์ที่เพิ่มมา -> ไม่หลุดไม่ซ้ำ)
drop_cols    = ["ID"] + ([] if USE_GENDER else ["Gender"])
RAW_FEATURES = [c for c in train.columns if c not in drop_cols + [TARGET]]
ENG_FEATURES = [c for c in full.columns if c not in train.columns]
ALL_FEATURES = RAW_FEATURES + ENG_FEATURES

X_all  = full.iloc[:len(train)].reset_index(drop=True)
X_test = full.iloc[len(train):].reset_index(drop=True)
y      = train[TARGET].values

print(f"{len(RAW_FEATURES)} raw + {len(ENG_FEATURES)} engineered = {len(ALL_FEATURES)} features")
print("engineered:", ", ".join(ENG_FEATURES))

24 raw + 13 engineered = 37 features
engineered: borrow_to_income, limit_to_income, exposure_to_income, limit_to_borrow, avg_loan, rent_to_income, income_per_dependent, children_share, residence_ratio, phone_count, occ_x_ind_freq, size_x_entity_freq, occ_x_emp_freq


## 3. โมเดลและ Cross-validation
- ใช้ **fold ชุดเดียวกัน** ทุกการทดลอง เพื่อให้เทียบกันได้ตรงๆ
- ทำนาย test ด้วยค่าเฉลี่ยของโมเดลจากทุก fold
- ไม่ใช้ `class_weight` เพราะ AUC ไม่ต้องการ
- ผลทุกการทดลองถูกเก็บใน `scoreboard` (ดูตารางสรุปในหัวข้อ 4)

In [48]:
folds      = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_all, y))
scoreboard = []        # เก็บผลทุกการทดลอง


def run_cv(fit_predict, features, name):
    oof, pred, fold_aucs = np.zeros(len(X_all)), np.zeros(len(X_test)), []
    for tr, va in folds:
        p_val, p_test = fit_predict(X_all.iloc[tr][features], y[tr],
                                    X_all.iloc[va][features], y[va],
                                    X_test[features])
        oof[va] = p_val
        pred   += p_test / len(folds)
        fold_aucs.append(roc_auc_score(y[va], p_val))

    row = dict(experiment=name, oof_auc=roc_auc_score(y, oof),
               fold_mean=np.mean(fold_aucs), fold_std=np.std(fold_aucs))
    scoreboard.append(row)
    print(f"{name:<28} OOF AUC {row['oof_auc']:.4f} | fold {row['fold_mean']:.4f} ± {row['fold_std']:.4f}")
    return oof, pred


def show_scoreboard():
    df = pd.DataFrame(scoreboard).set_index("experiment")
    df["diff_vs_first"] = df["oof_auc"] - df["oof_auc"].iloc[0]     # เทียบกับแถวแรก (baseline)
    display(df.round(4))

In [54]:
# ---------- LightGBM ----------
LGB_PARAMS = dict(
    n_estimators=5000, learning_rate=0.02,
    num_leaves=15, min_child_samples=200,                 # ต้นไม้เล็ก + ใบละหลายตัวอย่าง (signal อ่อน)
    subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
    reg_lambda=10.0, cat_smooth=50, min_data_per_group=200,
    random_state=SEED, n_jobs=-1, verbose=-1,
)

def fit_predict_lgb(X_tr, y_tr, X_va, y_va, X_te):
    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]


# ---------- CatBoost (ใช้เมื่อ USE_CATBOOST = True) ----------
def _cat_to_str(X):
    X = X.copy()
    for c in CAT_COLS:
        if c in X.columns:
            X[c] = X[c].astype(str)
    return X

def fit_predict_cat(X_tr, y_tr, X_va, y_va, X_te):
    cat_cols = [c for c in CAT_COLS if c in X_tr.columns]
    X_tr, X_va, X_te = _cat_to_str(X_tr), _cat_to_str(X_va), _cat_to_str(X_te)
    model = CatBoostClassifier(iterations=3000, learning_rate=0.03, depth=6, l2_leaf_reg=10,
                               eval_metric="AUC", random_seed=SEED, early_stopping_rounds=200,
                               cat_features=cat_cols, verbose=0, thread_count=-1)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
    return model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]

## 4. ทดลอง
**วิธีอ่านตาราง:** `diff_vs_first` คือส่วนต่างจากแถวแรก (raw features)
ถ้าต่างน้อยกว่า `fold_std` (ราว 0.009) ถือว่าเป็น noise -> ใช้ชุดที่เรียบง่ายกว่า

In [50]:
run_cv(fit_predict_lgb, RAW_FEATURES, "LGBM: raw features")          # baseline (ต้องรันเป็นอันแรก)
oof_lgb, pred_lgb = run_cv(fit_predict_lgb, ALL_FEATURES, "LGBM: raw + engineered")

final_models = {"LGBM": (oof_lgb, pred_lgb)}                       # โมเดลที่จะเอาไปทำ submission

if USE_CATBOOST:
    final_models["CatBoost"] = run_cv(fit_predict_cat, ALL_FEATURES, "CatBoost: raw + engineered")

LGBM: raw features           OOF AUC 0.6494 | fold 0.6494 ± 0.0116
LGBM: raw + engineered       OOF AUC 0.6511 | fold 0.6510 ± 0.0106


In [51]:
if RUN_ABLATION:
    feature_groups = {
        "ratios":          ["borrow_to_income", "limit_to_income", "exposure_to_income",
                            "limit_to_borrow", "avg_loan", "rent_to_income", "income_per_dependent"],
        "residence_ratio": ["residence_ratio"],
        "family":          ["children_share"],
        "phone_count":     ["phone_count"],
        "freq_encoding":   [c for c in ENG_FEATURES if c.endswith("_freq")],
    }
    for group, cols in feature_groups.items():
        run_cv(fit_predict_lgb, RAW_FEATURES + cols, f"LGBM: raw + {group}")

show_scoreboard()

,oof_auc,fold_mean,fold_std,diff_vs_first
experiment,,,,
LGBM: raw features,0.6494,0.6494,0.0116,0.0000
LGBM: raw + engineered,0.6511,0.6510,0.0106,0.0017


## 5. รวมผล + เลือก threshold
ถ้า metric ประเมินจาก **label 0/1** ที่ส่ง AUC = (TPR+TNR)/2 -> จุดตัดที่ดีที่สุดคือจุดที่ TPR−FPR สูงสุด (Youden's J) บน OOF
ไม่ใช่ค่าที่เดาหรือจูนจาก leaderboard ส่วนไฟล์ probability **ไม่ใช้ threshold นี้**

In [52]:
oof_final  = np.mean([oof  for oof, _  in final_models.values()], axis=0)
pred_final = np.mean([pred for _, pred in final_models.values()], axis=0)

fpr, tpr, thresholds = roc_curve(y, oof_final)
best      = np.argmax(tpr - fpr)
threshold = thresholds[best]

print(f"โมเดลที่ใช้     : {' + '.join(final_models)}")
print(f"OOF AUC (prob)  : {roc_auc_score(y, oof_final):.4f}   <- ไฟล์ probability ควรได้ใกล้เคียงนี้")
print(f"OOF AUC (label) : {0.5 + (tpr[best] - fpr[best]) / 2:.4f}   <- ไฟล์ 0/1 ควรได้ใกล้เคียงนี้")
print(f"threshold       : {threshold:.4f}")

โมเดลที่ใช้     : LGBM
OOF AUC (prob)  : 0.6511   <- ไฟล์ probability ควรได้ใกล้เคียงนี้
OOF AUC (label) : 0.6111   <- ไฟล์ 0/1 ควรได้ใกล้เคียงนี้
threshold       : 0.0882


## 6. Submission
สร้างไฟล์ 2 แบบ ตรวจรูปแบบกับ template ก่อนบันทึก

In [53]:
def make_submission(values, path):
    sub = pd.DataFrame({"ID": test["ID"].values, TARGET: values})
    sub = template[["ID"]].merge(sub, on="ID", how="left")        # เรียงตาม template
    assert list(sub.columns) == list(template.columns)
    assert len(sub) == len(template) and sub["ID"].is_unique
    assert sub[TARGET].notna().all()
    sub.to_csv(path, index=False)
    return sub

proba_file = f"submission_proba_{RUN_NAME}.csv"
label_file = f"submission_label_{RUN_NAME}.csv"
sub_proba  = make_submission(pred_final, proba_file)
sub_label  = make_submission((pred_final >= threshold).astype(int), label_file)

print(f"{proba_file:<32} ค่า {sub_proba[TARGET].min():.3f} ถึง {sub_proba[TARGET].max():.3f}   <- ส่งอันนี้ถ้ากติกาอนุญาตทศนิยม")
print(f"{label_file:<32} ทำนาย default {sub_label[TARGET].mean():.1%} ของ test   <- สำรอง (ต้อง 0/1)")
print("\nอย่าเปิดไฟล์ CSV แล้ว save ทับด้วย Excel: ID จะกลายเป็น 2.12E+08")

submission_proba_v7.csv          ค่า 0.027 ถึง 0.457   <- ส่งอันนี้ถ้ากติกาอนุญาตทศนิยม
submission_label_v7.csv          ทำนาย default 49.7% ของ test   <- สำรอง (ต้อง 0/1)

อย่าเปิดไฟล์ CSV แล้ว save ทับด้วย Excel: ID จะกลายเป็น 2.12E+08
